- Tool Decorators: Tag and document individual tools
- Tool Registry: Central storage of all available tools
- Action Registry: Curated sets of tools for specific agents

PENDIENTE

In [1]:
import inspect
from typing import get_type_hints, List, Dict, Callable

# Registros globales
tools: Dict[str, dict] = {}
tools_by_tag: Dict[str, List[str]] = {}

def get_json_type(python_type):
    """Convierte tipos de Python a tipos de JSON schema (simplificado)."""
    mapping = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array"}
    return mapping.get(python_type, "string")

def register_tool(tags=None):
    """Decorador que registra una función como herramienta."""
    def decorator(func):
        tool_name = func.__name__
        description = func.__doc__.strip() if func.__doc__ else "Sin descripción."
        
        signature = inspect.signature(func)
        type_hints = get_type_hints(func)
        params_schema = {"type": "object", "properties": {}, "required": []}
        
        for param_name, param in signature.parameters.items():
            param_type = type_hints.get(param_name, str)
            params_schema["properties"][param_name] = {"type": get_json_type(param_type)}
            if param.default == inspect.Parameter.empty:
                params_schema["required"].append(param_name)
        
        tools[tool_name] = {
            "description": description,
            "parameters": params_schema,
            "function": func,
            "tags": tags or []
        }
        
        for tag in (tags or []):
            tools_by_tag.setdefault(tag, []).append(tool_name)
        
        return func
    return decorator

In [2]:
@register_tool(tags=["file_operations", "read"])
def read_file(file_path: str) -> str:
    """Lee y devuelve el contenido de un archivo."""
    with open(file_path, 'r') as f:
        return f.read()

@register_tool(tags=["file_operations", "write"])
def write_file(file_path: str, content: str) -> None:
    """Escribe contenido en un archivo."""
    with open(file_path, 'w') as f:
        f.write(content)

@register_tool(tags=["database", "read"])
def query_database(query: str) -> str:
    """Ejecuta una consulta simulada a una base de datos."""
    return f"Resultados simulados para: {query}"

In [3]:
print("=== Todas las herramientas ===")
for name, meta in tools.items():
    print(f"\n {name}")
    print(f"   Descripción: {meta['description']}")
    print(f"   Parámetros: {meta['parameters']}")
    print(f"   Tags: {meta['tags']}")

print("\n=== Organizadas por tag ===")
for tag, herramientas in tools_by_tag.items():
    print(f"{tag}: {herramientas}")

=== Todas las herramientas ===

 read_file
   Descripción: Lee y devuelve el contenido de un archivo.
   Parámetros: {'type': 'object', 'properties': {'file_path': {'type': 'string'}}, 'required': ['file_path']}
   Tags: ['file_operations', 'read']

 write_file
   Descripción: Escribe contenido en un archivo.
   Parámetros: {'type': 'object', 'properties': {'file_path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['file_path', 'content']}
   Tags: ['file_operations', 'write']

 query_database
   Descripción: Ejecuta una consulta simulada a una base de datos.
   Parámetros: {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
   Tags: ['database', 'read']

=== Organizadas por tag ===
file_operations: ['read_file', 'write_file']
read: ['read_file', 'query_database']
write: ['write_file']
database: ['query_database']


In [4]:
class ActionRegistry:
    """Registro que filtra herramientas según los tags solicitados."""
    def __init__(self, tags: List[str]):
        self.tags = tags
        self.available_tools = {}
        for tag in tags:
            for tool_name in tools_by_tag.get(tag, []):
                self.available_tools[tool_name] = tools[tool_name]
    
    def list_tools(self):
        return list(self.available_tools.keys())

# Probemos con distintos agentes
read_only_registry = ActionRegistry(tags=["read"])
file_registry = ActionRegistry(tags=["file_operations"])

print("Agente 'read_only' puede usar:", read_only_registry.list_tools())
print("Agente 'file_operations' puede usar:", file_registry.list_tools())

Agente 'read_only' puede usar: ['read_file', 'query_database']
Agente 'file_operations' puede usar: ['read_file', 'write_file']
